# Lasso Regression - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import Lasso as SklearnLasso, Ridge as SklearnRidge
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is Lasso Regression?

Lasso (Least Absolute Shrinkage and Selection Operator) Regression is a linear regression technique that adds **L1 regularization** to the ordinary least squares (OLS) objective. It was introduced by Robert Tibshirani in 1996.

### Mathematical Formulation

#### Objective Function
$$\min_{\beta} \left\{ \frac{1}{2n} \sum_{i=1}^{n} (y_i - \beta_0 - \sum_{j=1}^{p} x_{ij}\beta_j)^2 + \alpha \sum_{j=1}^{p} |\beta_j| \right\}$$

Or equivalently:
$$\min_{\beta} \left\{ \frac{1}{2n} ||y - X\beta||_2^2 + \alpha ||\beta||_1 \right\}$$

Where:
- $||\cdot||_2^2$ is the squared L2 norm (RSS)
- $||\cdot||_1$ is the L1 norm (sum of absolute values)
- $\alpha$ is the regularization parameter

### L1 Regularization and Sparsity

The L1 penalty has a unique property: **it induces sparsity**. This means some coefficients become exactly zero, effectively performing automatic feature selection.

#### Why L1 Creates Sparsity

The constraint region for L1 (diamond-shaped) has corners along the axes. The optimal solution often occurs at these corners, where one or more coefficients are exactly zero.

### Comparison: Lasso vs Ridge

| Property | Lasso (L1) | Ridge (L2) |
|----------|------------|------------|
| Penalty | $\alpha\sum|\beta_j|$ | $\alpha\sum\beta_j^2$ |
| Constraint Shape | Diamond | Circle |
| Coefficient Behavior | Can be exactly zero | Shrink toward zero |
| Feature Selection | Automatic | No |
| Multicollinearity | Selects one feature | Distributes weight |
| Closed-form Solution | No | Yes |

### Coordinate Descent Algorithm

Unlike Ridge Regression, Lasso has no closed-form solution due to the non-differentiable L1 term. We use **Coordinate Descent**:

1. Initialize coefficients (e.g., zeros or OLS estimates)
2. For each coefficient $\beta_j$, holding others fixed:
   - Compute the partial residual: $r_j = y - \sum_{k \neq j} x_k \beta_k$
   - Update: $\beta_j = S(\rho_j, \alpha) / ||x_j||_2^2$
   
   Where $\rho_j = x_j^T r_j$ and $S$ is the soft-thresholding operator:
   $$S(\rho, \alpha) = \text{sign}(\rho) \max(|\rho| - \alpha, 0)$$

3. Repeat until convergence

### Soft-Thresholding Operator

The soft-thresholding function is key to Lasso:
$$S(\rho, \alpha) = \begin{cases}
\rho - \alpha & \text{if } \rho > \alpha \\
0 & \text{if } |\rho| \leq \alpha \\
\rho + \alpha & \text{if } \rho < -\alpha
\end{cases}$$

### Time Complexity
- Training: O(n_features * n_samples * n_iterations)
- Prediction: O(n_features * n_samples)

In [ ]:
# Visualize the soft-thresholding operator
def soft_threshold(rho, alpha):
    """Soft-thresholding operator."""
    return np.sign(rho) * np.maximum(np.abs(rho) - alpha, 0)

# Plot soft-thresholding for different alpha values
rho_values = np.linspace(-5, 5, 1000)
alphas = [0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Soft-thresholding function
for alpha in alphas:
    axes[0].plot(rho_values, soft_threshold(rho_values, alpha), 
                 label=f'alpha = {alpha}', linewidth=2)
axes[0].plot(rho_values, rho_values, 'k--', alpha=0.5, label='Identity (no penalty)')
axes[0].set_xlabel('rho (unconstrained coefficient)', fontsize=12)
axes[0].set_ylabel('Soft-thresholded coefficient', fontsize=12)
axes[0].set_title('Soft-Thresholding Operator', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].axvline(x=0, color='black', linewidth=0.5)

# L1 vs L2 constraint regions
theta = np.linspace(0, 2*np.pi, 100)
# L2 (circle)
l2_x = np.cos(theta)
l2_y = np.sin(theta)
# L1 (diamond)
l1_x = [1, 0, -1, 0, 1]
l1_y = [0, 1, 0, -1, 0]

axes[1].plot(l2_x, l2_y, 'b-', linewidth=2, label='L2 (Ridge) - Circle')
axes[1].plot(l1_x, l1_y, 'r-', linewidth=2, label='L1 (Lasso) - Diamond')
axes[1].set_xlabel('beta_1', fontsize=12)
axes[1].set_ylabel('beta_2', fontsize=12)
axes[1].set_title('L1 vs L2 Constraint Regions', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].axvline(x=0, color='black', linewidth=0.5)

# Add contour lines representing loss function
beta1 = np.linspace(-2, 2, 100)
beta2 = np.linspace(-2, 2, 100)
B1, B2 = np.meshgrid(beta1, beta2)
# Elliptical contours (representing typical RSS)
Z = (B1 - 0.8)**2 + 2*(B2 - 1.2)**2
axes[1].contour(B1, B2, Z, levels=[0.5, 1, 2, 3, 4], alpha=0.5, colors='gray')
axes[1].scatter([0.8], [1.2], color='green', s=100, marker='*', 
                label='OLS solution', zorder=5)

plt.tight_layout()
plt.show()

## 2. Implementation from Scratch <a id='implementation'></a>

In [ ]:
class LassoRegression:
    """
    Lasso Regression implementation from scratch using Coordinate Descent.
    
    Lasso adds L1 regularization to linear regression, which induces sparsity
    in the coefficients, effectively performing automatic feature selection.
    
    Parameters:
    -----------
    alpha : float, default=1.0
        Regularization strength. Must be positive.
        Larger values = more regularization = more sparsity.
    max_iter : int, default=1000
        Maximum number of iterations for coordinate descent.
    tol : float, default=1e-4
        Tolerance for convergence. The algorithm stops when the
        maximum coefficient change is less than tol.
    fit_intercept : bool, default=True
        Whether to calculate the intercept.
    warm_start : bool, default=False
        If True, reuse coefficients from previous fit as initialization.
    
    Attributes:
    -----------
    coef_ : ndarray of shape (n_features,)
        Estimated coefficients.
    intercept_ : float
        Intercept term.
    n_iter_ : int
        Number of iterations run.
    history_ : dict
        Training history (loss and coefficient changes).
    """
    
    def __init__(self, alpha=1.0, max_iter=1000, tol=1e-4, 
                 fit_intercept=True, warm_start=False):
        self.alpha = alpha
        self.max_iter = max_iter
        self.tol = tol
        self.fit_intercept = fit_intercept
        self.warm_start = warm_start
        
        # Attributes set during fitting
        self.coef_ = None
        self.intercept_ = 0.0
        self.n_iter_ = 0
        self.history_ = {'loss': [], 'coef_change': []}
    
    def _soft_threshold(self, rho, alpha):
        """
        Soft-thresholding operator for L1 regularization.
        
        S(rho, alpha) = sign(rho) * max(|rho| - alpha, 0)
        
        This is the key operation that induces sparsity in Lasso.
        """
        return np.sign(rho) * np.maximum(np.abs(rho) - alpha, 0)
    
    def _compute_loss(self, X, y):
        """
        Compute the Lasso objective function:
        (1/2n) * ||y - X*beta||^2 + alpha * ||beta||_1
        """
        n_samples = X.shape[0]
        residuals = y - self._predict_internal(X)
        mse = np.sum(residuals**2) / (2 * n_samples)
        l1_penalty = self.alpha * np.sum(np.abs(self.coef_))
        return mse + l1_penalty
    
    def _predict_internal(self, X):
        """Internal prediction without input validation."""
        return X @ self.coef_ + self.intercept_
    
    def fit(self, X, y):
        """
        Fit the Lasso model using Coordinate Descent.
        
        The algorithm iteratively updates each coefficient while holding
        the others fixed, using the soft-thresholding operator.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data.
        y : array-like of shape (n_samples,)
            Target values.
        
        Returns:
        --------
        self : LassoRegression
            Fitted model.
        """
        # Convert to numpy arrays
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()
        
        n_samples, n_features = X.shape
        
        # Initialize coefficients
        if not self.warm_start or self.coef_ is None:
            self.coef_ = np.zeros(n_features)
            self.intercept_ = 0.0
        
        # Reset history
        self.history_ = {'loss': [], 'coef_change': []}
        
        # Precompute column norms for efficiency
        # ||x_j||^2 is used in the denominator of the update
        col_norms_sq = np.sum(X**2, axis=0)
        
        # Coordinate descent iterations
        for iteration in range(self.max_iter):
            coef_old = self.coef_.copy()
            max_coef_change = 0.0
            
            # Update intercept (not regularized)
            if self.fit_intercept:
                self.intercept_ = np.mean(y - X @ self.coef_)
            
            # Update each coefficient
            for j in range(n_features):
                # Skip if column norm is zero (constant feature)
                if col_norms_sq[j] == 0:
                    continue
                
                # Compute partial residual (excluding feature j)
                # r_j = y - intercept - sum_{k!=j} X_k * beta_k
                residual = y - self.intercept_ - X @ self.coef_ + X[:, j] * self.coef_[j]
                
                # Compute rho_j = X_j^T * r_j / n
                rho_j = X[:, j] @ residual / n_samples
                
                # Apply soft-thresholding and normalize
                # beta_j = S(rho_j, alpha) / (||x_j||^2 / n)
                self.coef_[j] = self._soft_threshold(rho_j, self.alpha) / (col_norms_sq[j] / n_samples)
                
                # Track maximum coefficient change
                max_coef_change = max(max_coef_change, np.abs(self.coef_[j] - coef_old[j]))
            
            # Record history
            loss = self._compute_loss(X, y)
            self.history_['loss'].append(loss)
            self.history_['coef_change'].append(max_coef_change)
            
            # Check convergence
            if max_coef_change < self.tol:
                self.n_iter_ = iteration + 1
                break
        else:
            self.n_iter_ = self.max_iter
        
        return self
    
    def predict(self, X):
        """
        Predict using the linear model.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Samples to predict.
        
        Returns:
        --------
        y_pred : ndarray of shape (n_samples,)
            Predicted values.
        """
        X = np.asarray(X, dtype=np.float64)
        return X @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        """
        Return the coefficient of determination R^2.
        
        R^2 = 1 - SS_res / SS_tot
        """
        y = np.asarray(y).ravel()
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred)**2)
        ss_tot = np.sum((y - np.mean(y))**2)
        return 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
    
    def get_sparsity(self):
        """
        Return the fraction of zero coefficients.
        """
        return np.mean(self.coef_ == 0)
    
    def get_selected_features(self, threshold=1e-10):
        """
        Return indices of features with non-zero coefficients.
        
        Parameters:
        -----------
        threshold : float
            Coefficients with absolute value below this are considered zero.
        
        Returns:
        --------
        indices : ndarray
            Indices of selected (non-zero) features.
        """
        return np.where(np.abs(self.coef_) > threshold)[0]

In [ ]:
# Test the implementation on a simple example
print("Testing LassoRegression implementation...")
print("="*50)

# Create a simple dataset with known coefficients
np.random.seed(42)
n_samples, n_features = 100, 5
X_test = np.random.randn(n_samples, n_features)
true_coef = np.array([3.0, -2.0, 0.0, 1.5, 0.0])  # Note: 2 features are irrelevant
y_test = X_test @ true_coef + np.random.randn(n_samples) * 0.5

# Fit Lasso
lasso = LassoRegression(alpha=0.1, max_iter=1000, tol=1e-6)
lasso.fit(X_test, y_test)

print(f"True coefficients:     {true_coef}")
print(f"Estimated coefficients: {np.round(lasso.coef_, 4)}")
print(f"\nIntercept: {lasso.intercept_:.4f}")
print(f"R^2 score: {lasso.score(X_test, y_test):.4f}")
print(f"Iterations: {lasso.n_iter_}")
print(f"Sparsity: {lasso.get_sparsity()*100:.1f}% of coefficients are zero")
print(f"Selected features: {lasso.get_selected_features()}")

## 3. Training & Optimization <a id='training'></a>

In [ ]:
# Generate a synthetic dataset with some irrelevant features
# This is the ideal scenario for Lasso - some features are truly uninformative

np.random.seed(42)

# Dataset parameters
n_samples = 500
n_informative = 10   # Number of truly informative features
n_irrelevant = 20    # Number of irrelevant (noise) features
n_features = n_informative + n_irrelevant
noise_level = 1.0

# Generate features
X = np.random.randn(n_samples, n_features)

# Create true coefficients: only first n_informative are non-zero
true_coef = np.zeros(n_features)
true_coef[:n_informative] = np.random.uniform(-5, 5, n_informative)

# Generate target with noise
y = X @ true_coef + np.random.randn(n_samples) * noise_level

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features (important for Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dataset Summary")
print("="*50)
print(f"Total features: {n_features}")
print(f"  - Informative features: {n_informative}")
print(f"  - Irrelevant features: {n_irrelevant}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"\nTrue non-zero coefficients (first {n_informative}):")
print(np.round(true_coef[:n_informative], 3))

In [ ]:
# Train Lasso with different alpha values
alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0]
results = []

print("Training Lasso models with different alpha values...")
print("="*70)

for alpha in alphas:
    # Fit model
    lasso = LassoRegression(alpha=alpha, max_iter=2000, tol=1e-6)
    lasso.fit(X_train_scaled, y_train)
    
    # Evaluate
    train_mse = mean_squared_error(y_train, lasso.predict(X_train_scaled))
    test_mse = mean_squared_error(y_test, lasso.predict(X_test_scaled))
    n_nonzero = np.sum(np.abs(lasso.coef_) > 1e-10)
    r2_train = lasso.score(X_train_scaled, y_train)
    r2_test = lasso.score(X_test_scaled, y_test)
    
    results.append({
        'alpha': alpha,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'r2_train': r2_train,
        'r2_test': r2_test,
        'n_nonzero': n_nonzero,
        'n_iter': lasso.n_iter_,
        'coef': lasso.coef_.copy()
    })
    
    print(f"alpha={alpha:.3f}: Train MSE={train_mse:.4f}, Test MSE={test_mse:.4f}, "
          f"Non-zero coeffs={n_nonzero}/{n_features}, Iterations={lasso.n_iter_}")

# Convert to DataFrame for analysis
results_df = pd.DataFrame(results)

In [ ]:
# Visualize training convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train a model and show convergence
lasso_demo = LassoRegression(alpha=0.1, max_iter=500, tol=1e-8)
lasso_demo.fit(X_train_scaled, y_train)

# Loss curve
axes[0].plot(lasso_demo.history_['loss'], linewidth=2)
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('Loss (MSE + L1 penalty)', fontsize=12)
axes[0].set_title('Coordinate Descent Convergence', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Coefficient change
axes[1].plot(lasso_demo.history_['coef_change'], linewidth=2, color='orange')
axes[1].axhline(y=lasso_demo.tol, color='red', linestyle='--', label=f'Tolerance ({lasso_demo.tol})')
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Max Coefficient Change', fontsize=12)
axes[1].set_title('Convergence: Max Coefficient Change per Iteration', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
# Comprehensive alpha analysis
alphas_fine = np.logspace(-4, 1, 50)
mse_train_list = []
mse_test_list = []
n_nonzero_list = []
coef_matrix = []

print("Computing Lasso path across alpha values...")
for alpha in alphas_fine:
    lasso = LassoRegression(alpha=alpha, max_iter=2000, tol=1e-6)
    lasso.fit(X_train_scaled, y_train)
    
    mse_train_list.append(mean_squared_error(y_train, lasso.predict(X_train_scaled)))
    mse_test_list.append(mean_squared_error(y_test, lasso.predict(X_test_scaled)))
    n_nonzero_list.append(np.sum(np.abs(lasso.coef_) > 1e-10))
    coef_matrix.append(lasso.coef_.copy())

coef_matrix = np.array(coef_matrix)

In [ ]:
# Plot MSE vs alpha and number of non-zero coefficients vs alpha
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# MSE vs alpha
axes[0].plot(alphas_fine, mse_train_list, 'b-', linewidth=2, label='Train MSE')
axes[0].plot(alphas_fine, mse_test_list, 'r-', linewidth=2, label='Test MSE')
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha (regularization strength)', fontsize=12)
axes[0].set_ylabel('Mean Squared Error', fontsize=12)
axes[0].set_title('MSE vs Alpha', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Find optimal alpha (minimum test MSE)
optimal_idx = np.argmin(mse_test_list)
optimal_alpha = alphas_fine[optimal_idx]
axes[0].axvline(x=optimal_alpha, color='green', linestyle='--', 
                label=f'Optimal alpha={optimal_alpha:.4f}')
axes[0].legend()

# Non-zero coefficients vs alpha
axes[1].plot(alphas_fine, n_nonzero_list, 'g-', linewidth=2)
axes[1].axhline(y=n_informative, color='red', linestyle='--', 
                label=f'True # informative ({n_informative})')
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha (regularization strength)', fontsize=12)
axes[1].set_ylabel('Number of Non-zero Coefficients', fontsize=12)
axes[1].set_title('Sparsity vs Alpha', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Combined: Test MSE and sparsity
ax1 = axes[2]
ax2 = ax1.twinx()

line1, = ax1.plot(alphas_fine, mse_test_list, 'b-', linewidth=2, label='Test MSE')
line2, = ax2.plot(alphas_fine, n_nonzero_list, 'r-', linewidth=2, label='Non-zero coeffs')

ax1.set_xscale('log')
ax1.set_xlabel('Alpha', fontsize=12)
ax1.set_ylabel('Test MSE', color='blue', fontsize=12)
ax2.set_ylabel('Non-zero Coefficients', color='red', fontsize=12)
ax1.set_title('Trade-off: Accuracy vs Sparsity', fontsize=14)

lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right')
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nOptimal alpha (min test MSE): {optimal_alpha:.4f}")
print(f"Test MSE at optimal alpha: {mse_test_list[optimal_idx]:.4f}")
print(f"Non-zero coefficients at optimal alpha: {n_nonzero_list[optimal_idx]}")

In [ ]:
# Feature selection analysis
# Train model with optimal alpha
lasso_optimal = LassoRegression(alpha=optimal_alpha, max_iter=2000, tol=1e-6)
lasso_optimal.fit(X_train_scaled, y_train)

# Compare estimated vs true coefficients
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot of coefficients
x_pos = np.arange(n_features)
width = 0.35

# Scale true coefficients to standardized feature scale
scaled_true_coef = true_coef / scaler.scale_

bars1 = axes[0].bar(x_pos - width/2, scaled_true_coef, width, 
                    label='True Coefficients', color='blue', alpha=0.7)
bars2 = axes[0].bar(x_pos + width/2, lasso_optimal.coef_, width, 
                    label='Lasso Estimates', color='red', alpha=0.7)

axes[0].axvline(x=n_informative - 0.5, color='green', linestyle='--', 
                label='Informative/Irrelevant boundary')
axes[0].set_xlabel('Feature Index', fontsize=12)
axes[0].set_ylabel('Coefficient Value', fontsize=12)
axes[0].set_title('True vs Estimated Coefficients', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Feature importance (absolute coefficient values)
feature_importance = np.abs(lasso_optimal.coef_)
sorted_idx = np.argsort(feature_importance)[::-1]

colors = ['green' if idx < n_informative else 'red' for idx in sorted_idx]
axes[1].barh(range(n_features), feature_importance[sorted_idx], color=colors, alpha=0.7)
axes[1].set_yticks(range(n_features))
axes[1].set_yticklabels([f'Feature {i}' for i in sorted_idx])
axes[1].set_xlabel('Absolute Coefficient Value', fontsize=12)
axes[1].set_title('Feature Importance (Green=Informative, Red=Irrelevant)', fontsize=14)
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print feature selection accuracy
selected_features = lasso_optimal.get_selected_features()
true_informative = set(range(n_informative))
selected_set = set(selected_features)

true_positives = len(true_informative & selected_set)
false_positives = len(selected_set - true_informative)
false_negatives = len(true_informative - selected_set)

print("\nFeature Selection Analysis")
print("="*50)
print(f"True informative features: {n_informative}")
print(f"Selected features: {len(selected_features)}")
print(f"True Positives (correctly selected): {true_positives}")
print(f"False Positives (incorrectly selected): {false_positives}")
print(f"False Negatives (missed): {false_negatives}")
print(f"\nPrecision: {true_positives / len(selected_features) if len(selected_features) > 0 else 0:.3f}")
print(f"Recall: {true_positives / n_informative:.3f}")

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
# Coefficient paths (Lasso regularization path)
# This visualization shows how coefficients shrink to zero as alpha increases

fig, ax = plt.subplots(figsize=(12, 8))

# Plot coefficient paths
for j in range(n_features):
    if j < n_informative:
        # Informative features - solid lines
        ax.plot(alphas_fine, coef_matrix[:, j], linewidth=2, 
                label=f'Feature {j} (informative)')
    else:
        # Irrelevant features - dashed lines, lighter colors
        ax.plot(alphas_fine, coef_matrix[:, j], '--', linewidth=1, alpha=0.5)

ax.set_xscale('log')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axvline(x=optimal_alpha, color='green', linestyle=':', 
           label=f'Optimal alpha ({optimal_alpha:.4f})')
ax.set_xlabel('Alpha (regularization strength)', fontsize=12)
ax.set_ylabel('Coefficient Value', fontsize=12)
ax.set_title('Lasso Coefficient Paths\n(Solid=Informative, Dashed=Irrelevant)', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Implement Ridge for comparison
class RidgeRegression:
    """
    Ridge Regression for comparison with Lasso.
    Uses closed-form solution: beta = (X'X + alpha*I)^(-1) X'y
    """
    def __init__(self, alpha=1.0, fit_intercept=True):
        self.alpha = alpha
        self.fit_intercept = fit_intercept
        self.coef_ = None
        self.intercept_ = 0.0
    
    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()
        
        n_samples, n_features = X.shape
        
        if self.fit_intercept:
            self.intercept_ = np.mean(y)
            y_centered = y - self.intercept_
        else:
            y_centered = y
        
        # Closed-form solution
        I = np.eye(n_features)
        self.coef_ = np.linalg.solve(X.T @ X + self.alpha * n_samples * I, X.T @ y_centered)
        
        return self
    
    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        return X @ self.coef_ + self.intercept_
    
    def score(self, X, y):
        y = np.asarray(y).ravel()
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred)**2)
        ss_tot = np.sum((y - np.mean(y))**2)
        return 1 - (ss_res / ss_tot) if ss_tot > 0 else 0.0

In [ ]:
# Compare Lasso and Ridge coefficient paths
alphas_comparison = np.logspace(-4, 1, 50)

lasso_coefs = []
ridge_coefs = []

for alpha in alphas_comparison:
    # Lasso
    lasso = LassoRegression(alpha=alpha, max_iter=2000, tol=1e-6)
    lasso.fit(X_train_scaled, y_train)
    lasso_coefs.append(lasso.coef_.copy())
    
    # Ridge
    ridge = RidgeRegression(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    ridge_coefs.append(ridge.coef_.copy())

lasso_coefs = np.array(lasso_coefs)
ridge_coefs = np.array(ridge_coefs)

In [ ]:
# Side-by-side comparison of coefficient paths
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Lasso paths
for j in range(min(10, n_features)):  # Plot first 10 features for clarity
    color = 'C' + str(j % 10)
    axes[0].plot(alphas_comparison, lasso_coefs[:, j], color=color, 
                 linewidth=2, label=f'Feature {j}')

axes[0].set_xscale('log')
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].set_xlabel('Alpha', fontsize=12)
axes[0].set_ylabel('Coefficient Value', fontsize=12)
axes[0].set_title('LASSO Coefficient Paths\n(Coefficients become exactly zero)', fontsize=14)
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[0].grid(True, alpha=0.3)

# Ridge paths
for j in range(min(10, n_features)):
    color = 'C' + str(j % 10)
    axes[1].plot(alphas_comparison, ridge_coefs[:, j], color=color, 
                 linewidth=2, label=f'Feature {j}')

axes[1].set_xscale('log')
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_xlabel('Alpha', fontsize=12)
axes[1].set_ylabel('Coefficient Value', fontsize=12)
axes[1].set_title('RIDGE Coefficient Paths\n(Coefficients shrink but never reach zero)', fontsize=14)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sparsity comparison
lasso_nonzero = [np.sum(np.abs(c) > 1e-10) for c in lasso_coefs]
ridge_nonzero = [np.sum(np.abs(c) > 1e-10) for c in ridge_coefs]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(alphas_comparison, lasso_nonzero, 'b-', linewidth=2, label='Lasso')
ax.plot(alphas_comparison, ridge_nonzero, 'r-', linewidth=2, label='Ridge')
ax.axhline(y=n_informative, color='green', linestyle='--', 
           label=f'True # informative ({n_informative})')

ax.set_xscale('log')
ax.set_xlabel('Alpha', fontsize=12)
ax.set_ylabel('Number of Non-zero Coefficients', fontsize=12)
ax.set_title('Sparsity: Lasso vs Ridge', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key Observation:")
print("-" * 50)
print("Lasso produces true zeros (sparsity), making it ideal for feature selection.")
print("Ridge only shrinks coefficients toward zero but never reaches exactly zero.")

In [ ]:
# Residual analysis for the optimal Lasso model
y_pred_train = lasso_optimal.predict(X_train_scaled)
y_pred_test = lasso_optimal.predict(X_test_scaled)
residuals_train = y_train - y_pred_train
residuals_test = y_test - y_pred_test

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Predicted vs Actual
axes[0, 0].scatter(y_test, y_pred_test, alpha=0.6, edgecolors='black', linewidths=0.5)
min_val, max_val = min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0, 0].set_xlabel('Actual Values', fontsize=12)
axes[0, 0].set_ylabel('Predicted Values', fontsize=12)
axes[0, 0].set_title('Predicted vs Actual (Test Set)', fontsize=14)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Residuals vs Predicted
axes[0, 1].scatter(y_pred_test, residuals_test, alpha=0.6, edgecolors='black', linewidths=0.5)
axes[0, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Predicted Values', fontsize=12)
axes[0, 1].set_ylabel('Residuals', fontsize=12)
axes[0, 1].set_title('Residuals vs Predicted', fontsize=14)
axes[0, 1].grid(True, alpha=0.3)

# Residual histogram
axes[1, 0].hist(residuals_test, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Residuals', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)
axes[1, 0].set_title('Residual Distribution', fontsize=14)
axes[1, 0].grid(True, alpha=0.3)

# Q-Q plot
from scipy import stats
stats.probplot(residuals_test, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot (Normality Check)', fontsize=14)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print metrics
print("\nModel Performance Metrics")
print("="*50)
print(f"R^2 Score (Train): {lasso_optimal.score(X_train_scaled, y_train):.4f}")
print(f"R^2 Score (Test):  {lasso_optimal.score(X_test_scaled, y_test):.4f}")
print(f"MSE (Train): {mean_squared_error(y_train, y_pred_train):.4f}")
print(f"MSE (Test):  {mean_squared_error(y_test, y_pred_test):.4f}")
print(f"MAE (Test):  {mean_absolute_error(y_test, y_pred_test):.4f}")
print(f"RMSE (Test): {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use Lasso Regression

#### Best Use Cases:

1. **Feature Selection is Needed**
   - You have many features and suspect some are irrelevant
   - You want automatic feature selection
   - You need an interpretable sparse model

2. **High-Dimensional Data (p >> n)**
   - More features than samples
   - Genomics, text classification, sensor data
   - When traditional regression fails

3. **Sparse True Model**
   - When you believe only a few features truly matter
   - Scientific applications where parsimony is valued

4. **Reducing Model Complexity**
   - For deployment with limited resources
   - When simpler models are preferred

#### When NOT to Use Lasso:

1. **All Features Are Important**
   - If all features contribute, Lasso may incorrectly zero out some
   - Use Ridge Regression instead

2. **Highly Correlated Features (Multicollinearity)**
   - Lasso tends to arbitrarily select one from a group of correlated features
   - Consider Elastic Net (combines L1 and L2)

3. **Group Selection Needed**
   - When you want to select/exclude groups of features together
   - Use Group Lasso instead

4. **Stable Feature Selection Required**
   - Lasso selection can be unstable with correlated features
   - Consider Stability Selection or Elastic Net

### Alpha Selection Guidelines

| Alpha Range | Effect | Typical Use |
|-------------|--------|-------------|
| Very small (< 0.01) | Minimal regularization, nearly OLS | When you don't want much sparsity |
| Small (0.01 - 0.1) | Moderate sparsity | General purpose |
| Medium (0.1 - 1.0) | Significant sparsity | Feature selection focus |
| Large (> 1.0) | Heavy sparsity | Very few features |

### Alpha Selection Methods:

1. **Cross-Validation** (Most common)
   - Use LassoCV or manual k-fold CV
   - Select alpha that minimizes CV error

2. **Information Criteria**
   - AIC, BIC for model selection
   - Penalize model complexity

3. **One Standard Error Rule**
   - Select largest alpha within one SE of minimum CV error
   - Produces sparser, more stable models

### Comparison Summary

| Aspect | Lasso | Ridge | Elastic Net |
|--------|-------|-------|-------------|
| Penalty | L1 | L2 | L1 + L2 |
| Feature Selection | Yes | No | Yes |
| Handles Correlated Features | Poorly | Well | Better |
| Solution | Iterative | Closed-form | Iterative |
| Sparsity | Yes | No | Yes |
| Stability | Lower | Higher | Medium |

### Data Requirements

1. **Standardize Features**: Lasso is sensitive to feature scales
2. **Handle Missing Values**: Impute or remove before fitting
3. **Sample Size**: Works even when p > n
4. **Outliers**: Consider robust preprocessing

In [ ]:
# Demonstration: Impact of feature standardization
np.random.seed(42)

# Create data with different feature scales
n = 200
X_scale_demo = np.column_stack([
    np.random.randn(n) * 1,      # Scale 1
    np.random.randn(n) * 100,    # Scale 100
    np.random.randn(n) * 0.01    # Scale 0.01
])
true_coef_demo = np.array([2.0, 0.02, 200.0])  # Coefficients adjusted for scale
y_scale_demo = X_scale_demo @ true_coef_demo + np.random.randn(n) * 0.5

# Fit without standardization
lasso_unstd = LassoRegression(alpha=0.1, max_iter=2000)
lasso_unstd.fit(X_scale_demo, y_scale_demo)

# Fit with standardization
scaler_demo = StandardScaler()
X_scale_demo_std = scaler_demo.fit_transform(X_scale_demo)
lasso_std = LassoRegression(alpha=0.1, max_iter=2000)
lasso_std.fit(X_scale_demo_std, y_scale_demo)

print("Impact of Feature Standardization")
print("="*60)
print(f"\nFeature scales: [1, 100, 0.01]")
print(f"True coefficients (in original scale): {true_coef_demo}")
print(f"\nWithout standardization:")
print(f"  Estimated: {np.round(lasso_unstd.coef_, 4)}")
print(f"  R^2: {lasso_unstd.score(X_scale_demo, y_scale_demo):.4f}")
print(f"\nWith standardization:")
print(f"  Estimated (standardized scale): {np.round(lasso_std.coef_, 4)}")
print(f"  R^2: {lasso_std.score(X_scale_demo_std, y_scale_demo):.4f}")
print(f"\nNote: Standardization ensures fair comparison between features.")

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare our implementation with sklearn
from sklearn.linear_model import Lasso as SklearnLasso

# Test on the same data with the same alpha
test_alpha = 0.1

# Our implementation
our_lasso = LassoRegression(alpha=test_alpha, max_iter=5000, tol=1e-8)
our_lasso.fit(X_train_scaled, y_train)

# sklearn implementation
sklearn_lasso = SklearnLasso(alpha=test_alpha, max_iter=5000, tol=1e-8)
sklearn_lasso.fit(X_train_scaled, y_train)

print("Comparison: Our Implementation vs sklearn")
print("="*60)
print(f"\nAlpha: {test_alpha}")
print(f"\nOur Implementation:")
print(f"  Train R^2: {our_lasso.score(X_train_scaled, y_train):.6f}")
print(f"  Test R^2:  {our_lasso.score(X_test_scaled, y_test):.6f}")
print(f"  Train MSE: {mean_squared_error(y_train, our_lasso.predict(X_train_scaled)):.6f}")
print(f"  Test MSE:  {mean_squared_error(y_test, our_lasso.predict(X_test_scaled)):.6f}")
print(f"  Non-zero coeffs: {np.sum(np.abs(our_lasso.coef_) > 1e-10)}")
print(f"  Iterations: {our_lasso.n_iter_}")

print(f"\nsklearn Implementation:")
print(f"  Train R^2: {sklearn_lasso.score(X_train_scaled, y_train):.6f}")
print(f"  Test R^2:  {sklearn_lasso.score(X_test_scaled, y_test):.6f}")
print(f"  Train MSE: {mean_squared_error(y_train, sklearn_lasso.predict(X_train_scaled)):.6f}")
print(f"  Test MSE:  {mean_squared_error(y_test, sklearn_lasso.predict(X_test_scaled)):.6f}")
print(f"  Non-zero coeffs: {np.sum(np.abs(sklearn_lasso.coef_) > 1e-10)}")
print(f"  Iterations: {sklearn_lasso.n_iter_}")

In [ ]:
# Coefficient comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot comparison
x_pos = np.arange(n_features)
width = 0.35

axes[0].bar(x_pos - width/2, our_lasso.coef_, width, label='Our Implementation', alpha=0.7)
axes[0].bar(x_pos + width/2, sklearn_lasso.coef_, width, label='sklearn', alpha=0.7)
axes[0].set_xlabel('Feature Index', fontsize=12)
axes[0].set_ylabel('Coefficient Value', fontsize=12)
axes[0].set_title('Coefficient Comparison', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot (correlation)
axes[1].scatter(sklearn_lasso.coef_, our_lasso.coef_, alpha=0.7, s=100, edgecolors='black')
min_coef = min(sklearn_lasso.coef_.min(), our_lasso.coef_.min())
max_coef = max(sklearn_lasso.coef_.max(), our_lasso.coef_.max())
axes[1].plot([min_coef, max_coef], [min_coef, max_coef], 'r--', linewidth=2, 
             label='Perfect agreement')
axes[1].set_xlabel('sklearn Coefficients', fontsize=12)
axes[1].set_ylabel('Our Coefficients', fontsize=12)
axes[1].set_title('Coefficient Correlation', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute correlation
correlation = np.corrcoef(our_lasso.coef_, sklearn_lasso.coef_)[0, 1]
max_diff = np.max(np.abs(our_lasso.coef_ - sklearn_lasso.coef_))
print(f"\nCoefficient correlation: {correlation:.6f}")
print(f"Maximum coefficient difference: {max_diff:.6f}")

In [ ]:
# Performance comparison across multiple alpha values
alphas_test = np.logspace(-3, 0, 20)

our_r2_scores = []
sklearn_r2_scores = []
our_times = []
sklearn_times = []

import time

for alpha in alphas_test:
    # Our implementation
    start = time.time()
    our_model = LassoRegression(alpha=alpha, max_iter=2000, tol=1e-6)
    our_model.fit(X_train_scaled, y_train)
    our_times.append(time.time() - start)
    our_r2_scores.append(our_model.score(X_test_scaled, y_test))
    
    # sklearn
    start = time.time()
    sk_model = SklearnLasso(alpha=alpha, max_iter=2000, tol=1e-6)
    sk_model.fit(X_train_scaled, y_train)
    sklearn_times.append(time.time() - start)
    sklearn_r2_scores.append(sk_model.score(X_test_scaled, y_test))

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R^2 comparison
axes[0].plot(alphas_test, our_r2_scores, 'b-o', label='Our Implementation', linewidth=2, markersize=6)
axes[0].plot(alphas_test, sklearn_r2_scores, 'r--s', label='sklearn', linewidth=2, markersize=6)
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha', fontsize=12)
axes[0].set_ylabel('Test R^2 Score', fontsize=12)
axes[0].set_title('R^2 Score Comparison Across Alpha Values', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Time comparison
axes[1].plot(alphas_test, our_times, 'b-o', label='Our Implementation', linewidth=2, markersize=6)
axes[1].plot(alphas_test, sklearn_times, 'r--s', label='sklearn', linewidth=2, markersize=6)
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha', fontsize=12)
axes[1].set_ylabel('Training Time (seconds)', fontsize=12)
axes[1].set_title('Training Time Comparison', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAverage time - Our Implementation: {np.mean(our_times)*1000:.2f} ms")
print(f"Average time - sklearn: {np.mean(sklearn_times)*1000:.2f} ms")
print(f"\nNote: sklearn uses optimized Cython/Fortran code and additional optimizations.")

In [ ]:
# Demonstrate sklearn's LassoCV for automatic alpha selection
from sklearn.linear_model import LassoCV

# LassoCV automatically selects optimal alpha using cross-validation
lasso_cv = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

print("sklearn LassoCV Results")
print("="*50)
print(f"Optimal alpha (via CV): {lasso_cv.alpha_:.6f}")
print(f"Test R^2: {lasso_cv.score(X_test_scaled, y_test):.4f}")
print(f"Non-zero coefficients: {np.sum(np.abs(lasso_cv.coef_) > 1e-10)}")
print(f"\nCompare with our optimal alpha: {optimal_alpha:.6f}")

## Summary & Key Takeaways

### What We Learned:

1. **Lasso Regression** adds L1 regularization to linear regression, inducing sparsity
2. **Coordinate Descent** is the standard algorithm for solving Lasso (no closed-form solution)
3. **Soft-thresholding** is the key operation that sets coefficients to exactly zero
4. **Feature Selection** is automatic - Lasso identifies relevant features

### Key Differences from Ridge:
- Lasso produces sparse solutions (exact zeros)
- Ridge shrinks but doesn't eliminate coefficients
- Use Lasso for feature selection, Ridge when all features matter

### Practical Guidelines:
- Always standardize features before applying Lasso
- Use cross-validation to select alpha
- Consider Elastic Net for correlated features
- Lasso is ideal when you believe in a sparse true model

### When to Choose Lasso:
- Feature selection is needed
- High-dimensional data (p >> n)
- Interpretability with few features
- Sparse underlying model

### Next Steps:
- Explore Elastic Net (combines L1 and L2)
- Implement Group Lasso for grouped feature selection
- Try Adaptive Lasso for oracle properties
- Use cross-validation for robust alpha selection